# Install Packages

In [30]:
!pip install mlflow xgboost -q

# Import Libraries

In [31]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

# Recreate Dataset

In [32]:
np.random.seed(42)

n_samples = 10000

temperature = np.random.normal(75,15,n_samples)
vibration = np.random.normal(0.5,0.2,n_samples)
pressure = np.random.normal(100,20,n_samples)
rpm = np.random.normal(1500,200,n_samples)
age_days = np.random.randint(0,365,n_samples)

failure_score = (
    (temperature > 90)*0.3 +
    (vibration > 0.8)*0.3 +
    (pressure > 130)*0.2 +
    (age_days > 300)*0.2
)

failure_prob = failure_score + np.random.normal(0,0.1,n_samples)

failure = (failure_prob > 0.35).astype(int)

data = pd.DataFrame({
    'temperature': temperature,
    'vibration': vibration,
    'pressure': pressure,
    'rpm': rpm,
    'age_days': age_days,
    'failure': failure
})

print(data.shape)

(10000, 6)


# Split Data

In [33]:
X = data.drop('failure', axis=1)
y = data['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(8000, 5)
(2000, 5)


# Scale Data

In [34]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling Complete")

Scaling Complete


# Train Logistic Regression

In [35]:
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:,1]

lr_auc = roc_auc_score(y_test, lr_prob)

print("Logistic Regression ROC AUC:", lr_auc)

Logistic Regression ROC AUC: 0.8460205629529749


# Train Random Forest

In [36]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC AUC:", rf_auc)

Random Forest ROC AUC: 0.9421900106747569


# Train XGBoost

In [37]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train)

xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:,1]

xgb_auc = roc_auc_score(y_test, xgb_prob)

print("XGBoost ROC AUC:", xgb_auc)

XGBoost ROC AUC: 0.9413405247485813


# Compare Models

In [38]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "ROC_AUC": [
        lr_auc,
        rf_auc,
        xgb_auc
    ]
})

results = results.sort_values(
    "ROC_AUC",
    ascending=False
)

results

,Model,ROC_AUC
1,Random Forest,0.942190
2,XGBoost,0.941341
0,Logistic Regression,0.846021


# Register Best Model

In [39]:
model_name = "PredictiveMaintenance"

registry = {
    model_name: {
        "version": 1,
        "model": xgb_model,
        "stage": "None"
    }
}

print("Registered:")
print(model_name)
print("Version:", registry[model_name]["version"])

Registered:
PredictiveMaintenance
Version: 1


# Add Documentation

In [40]:
registry[model_name]["description"] = """
XGBoost model for predictive maintenance.

Features:
temperature
vibration
pressure
rpm
age_days
"""

registry[model_name]["tags"] = {
    "validation_status": "passed",
    "team": "data-science",
    "framework": "xgboost"
}

print("Documentation Added")

Documentation Added


# Move to Staging

In [41]:
registry[model_name]["stage"] = "Staging"

print("Model moved to Staging")

Model moved to Staging


# Test Staging Model

In [42]:
test_data = pd.DataFrame({
    'temperature':[95],
    'vibration':[0.9],
    'pressure':[135],
    'rpm':[1500],
    'age_days':[320]
})

test_scaled = scaler.transform(test_data)

prediction = xgb_model.predict(test_scaled)

print("Prediction:", prediction[0])

if prediction[0] == 1:
    print("FAILURE LIKELY")
else:
    print("NO FAILURE")

Prediction: 1
FAILURE LIKELY


# Promote to Production

In [43]:
registry[model_name]["stage"] = "Production"

print("Version 1 is now in Production")

Version 1 is now in Production


# Production Inference Function

In [44]:
def predict_equipment_failure(
    temperature,
    vibration,
    pressure,
    rpm,
    age_days
):

    input_data = pd.DataFrame([{
        'temperature': temperature,
        'vibration': vibration,
        'pressure': pressure,
        'rpm': rpm,
        'age_days': age_days
    }])

    input_scaled = scaler.transform(input_data)

    prediction = xgb_model.predict(input_scaled)[0]

    return {
        "will_fail": bool(prediction),
        "recommendation":
        "Schedule maintenance"
        if prediction
        else
        "Normal operation"
    }

# Test Three Scenarios

In [45]:
scenarios = [

    {
        "name":"Normal",
        "temp":70,
        "vib":0.4,
        "press":95,
        "rpm":1500,
        "age":100
    },

    {
        "name":"High Risk",
        "temp":95,
        "vib":0.9,
        "press":135,
        "rpm":1500,
        "age":320
    },

    {
        "name":"Medium",
        "temp":85,
        "vib":0.6,
        "press":110,
        "rpm":1500,
        "age":200
    }
]

for s in scenarios:

    result = predict_equipment_failure(
        s["temp"],
        s["vib"],
        s["press"],
        s["rpm"],
        s["age"]
    )

    print(
        s["name"],
        "→",
        result["recommendation"]
    )

Normal → Normal operation
High Risk → Schedule maintenance
Medium → Normal operation


# Register Version 2

In [46]:
registry["PredictiveMaintenance_v2"] = {
    "version": 2,
    "model": rf_model,
    "stage": "None"
}

print("Version 2 Registered")

Version 2 Registered


# Rollback

In [47]:
production_version = 1

print("Rolling Back...")

print(
    "Production now uses Version",
    production_version
)

Rolling Back...
Production now uses Version 1
